<a href="https://colab.research.google.com/github/Airplane356/video-background-remover/blob/main/MatAnyone2_Video_Background_Remover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
!git clone https://github.com/pq-yang/MatAnyone2.git
%cd MatAnyone2
!pip install -e . -q

!pip install git+https://github.com/facebookresearch/sam2.git -q

!pip install ultralytics -q

!pip install torch torchvision -q

!pip install "imageio>=2.33,!=2.35.0" "huggingface-hub>=1.5.0,<2.0" -q

SAM2_PATH_LOCAL = '/content/pretrained_models/sam2_hiera_small.pt'

os.makedirs('/content/pretrained_models', exist_ok=True)

if not os.path.exists(SAM2_PATH_LOCAL):
    !wget https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt \
        -O {SAM2_PATH_LOCAL}
    print(f'✅ SAM2 downloaded: {os.path.getsize(SAM2_PATH_LOCAL)/1e6:.1f} MB')
else:
    print('✅ SAM2 weights already present')

print('\n✅ All dependencies installed.')

In [ ]:
import os, requests, tqdm

MA2_PATH = 'pretrained_models/matanyone2.pth'
MA2_URL  = 'https://github.com/pq-yang/MatAnyone2/releases/download/v1.0.0/matanyone2.pth'

if os.path.exists(MA2_PATH):
    os.remove(MA2_PATH)
    print('Removed corrupt checkpoint')

os.makedirs('pretrained_models', exist_ok=True)

response = requests.get(MA2_URL, stream=True)
total = int(response.headers.get('content-length', 0))

with open(MA2_PATH, 'wb') as f, tqdm.tqdm(
    desc='matanyone2.pth',
    total=total,
    unit='B',
    unit_scale=True,
    unit_divisor=1024,
) as bar:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
        bar.update(len(chunk))

print(f'Downloaded: {os.path.getsize(MA2_PATH) / 1e6:.1f} MB')

In [ ]:
# ──── USER CONFIG ──────────────────────────────────────────────────────────────────────
import sam2 as _sam2, os as _os

INPUT_VIDEO   = '/content/video.mp4'
OUTPUT_DIR    = '/content/matanyone2_out'

OUTPUT_MODE   = 'greenscreen'

CUT_THRESHOLD = 0.30

MAX_SIZE      = 1080

N_WARMUP      = 20

R_ERODE       = 10
R_DILATE      = 10

YOLO_CONF     = 0.4

SAM2_CFG      = _os.path.join(_os.path.dirname(_sam2.__file__), 'configs/sam2/sam2_hiera_s.yaml')
SAM2_PATH     = '/content/pretrained_models/sam2_hiera_small.pt'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config loaded.')

In [ ]:
import torch
import numpy as np
import cv2
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# ── MatAnyone 2 ───────────────────────────────────────────────────────────────
from hydra.core.global_hydra import GlobalHydra
from matanyone2.utils.get_default_model import get_matanyone2_model

GlobalHydra.instance().clear()
matanyone2_model = get_matanyone2_model(MA2_PATH, device)
print('✅ MatAnyone 2 loaded')

# ── SAM 2 image predictor ─────────────────────────────────────────────────────
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from omegaconf import OmegaConf
from hydra.utils import instantiate
import sam2 as _sam2

cfg_path = os.path.join(os.path.dirname(_sam2.__file__), 'configs/sam2/sam2_hiera_s.yaml')
cfg = OmegaConf.load(cfg_path)

model = instantiate(cfg.model, _recursive_=True)
state_dict = torch.load(SAM2_PATH, map_location=device)
model.load_state_dict(state_dict['model'], strict=False)
model = model.to(device).eval()
sam2_predictor = SAM2ImagePredictor(model)
print('✅ SAM 2 loaded')

# ── YOLOv8n person detector ───────────────────────────────────────────────────
from ultralytics import YOLO
yolo_model = YOLO('yolov8n.pt')
print('✅ YOLOv8n loaded')

In [ ]:
import torch.nn.functional as F
import imageio
from tqdm import tqdm
from matanyone2.inference.inference_core import InferenceCore
from matanyone2.utils.inference_utils import gen_dilate, gen_erosion


# ─────────────────────────────────────────────────────────────────────────────
#  1. SCENE-CUT DETECTION
#     Uses per-channel histogram difference between consecutive frames.
#     RVM's implicit background model breaks at cuts — we avoid that entirely
#     by segmenting the video into shots first.
# ─────────────────────────────────────────────────────────────────────────────
def detect_scene_cuts(video_path: str, threshold: float = 0.35) -> list[int]:
    """
    Returns a list of frame indices where a scene cut was detected.
    Frame 0 is always included as the first shot boundary.

    Method: normalised histogram difference across all three HSV channels.
    HSV is used instead of RGB because it's more robust to gradual lighting
    changes — hue and saturation shift is a stronger cut signal than brightness.
    """
    cap = cv2.VideoCapture(video_path)
    cut_frames = [0]
    prev_hist = None
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        hist = [
            cv2.calcHist([hsv], [c], None, [64], [0, 256])
            for c in range(3)
        ]
        for h in hist:
            cv2.normalize(h, h)

        if prev_hist is not None:
            diffs = [
                cv2.compareHist(prev_hist[c], hist[c], cv2.HISTCMP_BHATTACHARYYA)
                for c in range(3)
            ]
            score = float(np.mean(diffs))
            if score > threshold:
                cut_frames.append(frame_idx)

        prev_hist = hist
        frame_idx += 1

    cap.release()
    return cut_frames


# ─────────────────────────────────────────────────────────────────────────────
#  2. AUTO FOREGROUND MASK GENERATION (per shot, on first frame only)
#     RVM's insight: train simultaneously on segmentation + matting so the
#     model learns implicit foreground priors. We replicate the foreground
#     *detection* step from RVM's approach (semantic understanding of humans)
#     using YOLO, but only run it once per shot — not as a continuous tracker.
#     SAM 2 then converts the bounding box into a pixel-precise mask.
# ─────────────────────────────────────────────────────────────────────────────
def get_foreground_mask(
    frame_bgr: np.ndarray,
    yolo_conf: float = 0.4,
) -> np.ndarray | None:
    h, w = frame_bgr.shape[:2]
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    # ── Step 1: YOLO person detection ─────────────────────────────────────────
    yolo_results = yolo_model(
        frame_rgb,
        conf=yolo_conf,
        classes=[0],   # class 0 = person in COCO
        verbose=False,
    )[0]

    boxes_xyxy = []
    if yolo_results.boxes is not None and len(yolo_results.boxes) > 0:
        raw_boxes = yolo_results.boxes.xyxy.cpu().numpy()   # [N, 4]
        confs     = yolo_results.boxes.conf.cpu().numpy()   # [N]

        # Score each box: favour large, central detections
        scored = []
        cx_frame, cy_frame = w / 2, h / 2
        for box, conf in zip(raw_boxes, confs):
            x1, y1, x2, y2 = box
            bw, bh = x2 - x1, y2 - y1
            cx_box, cy_box = (x1 + x2) / 2, (y1 + y2) / 2
            centrality = 1.0 - (
                abs(cx_box - cx_frame) / cx_frame * 0.5 +
                abs(cy_box - cy_frame) / cy_frame * 0.5
            )
            area_frac = (bw * bh) / (w * h)
            score = float(conf) * (1 + centrality) * (1 + area_frac * 3)
            scored.append((score, box))

        best_score = max(s for s, _ in scored)
        boxes_xyxy = [
            b for s, b in scored if s >= 0.6 * best_score
        ]

    # ── Step 2: SAM 2 mask prediction ─────────────────────────────────────────
    sam2_predictor.set_image(frame_rgb)

    if boxes_xyxy:
        # Multi-box prompt: get one mask per person box, then union
        combined_mask = np.zeros((h, w), dtype=np.uint8)
        for box in boxes_xyxy:
            masks, scores, _ = sam2_predictor.predict(
                box=np.array(box),
                multimask_output=False,
            )
            best = masks[np.argmax(scores)].astype(np.uint8) * 255
            combined_mask = cv2.bitwise_or(combined_mask, best)
        return combined_mask

    else:
        # Fallback: no person detected — use SAM2 automatic mask generator
        print('  [WARN] No person detected by YOLO — using SAM2 largest-mask fallback')
        from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
        amg = SAM2AutomaticMaskGenerator(
            sam2_model,
            points_per_side=16,
            pred_iou_thresh=0.7,
            stability_score_thresh=0.8,
        )
        auto_masks = amg.generate(frame_rgb)
        if not auto_masks:
            return None

        auto_masks.sort(key=lambda m: m['area'], reverse=True)
        for m in auto_masks:
            frac = m['area'] / (h * w)
            if 0.05 < frac < 0.85:
                return (m['segmentation'].astype(np.uint8) * 255)
        return None


# ─────────────────────────────────────────────────────────────────────────────
#  3. MATANYONE 2 SHOT PROCESSOR
#     Runs MatAnyone 2 on a list of BGR frames using a given first-frame mask.
#     Returns list of (alpha, foreground_rgba) numpy arrays.
# ─────────────────────────────────────────────────────────────────────────────
def run_matanyone2_on_shot(
    frames_bgr: list[np.ndarray],
    first_frame_mask: np.ndarray,
    n_warmup: int = 10,
    r_erode: int = 10,
    r_dilate: int = 10,
) -> list[dict]:
    """
    Returns a list of dicts {'pha': np.ndarray [H,W,1], 'fgr': np.ndarray [H,W,3]}
    for each frame in frames_bgr (same length).
    """
    processor = InferenceCore(matanyone2_model, cfg=matanyone2_model.cfg)

    # Convert BGR frames to RGB float tensors [T, C, H, W]
    vframes = torch.stack([
        torch.from_numpy(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)).permute(2, 0, 1)
        for f in frames_bgr
    ]).float()

    #  resize
    h_orig, w_orig = vframes.shape[-2:]
    resized = False
    if MAX_SIZE > 0:
        min_side = min(h_orig, w_orig)
        if min_side > MAX_SIZE:
            scale = MAX_SIZE / min_side
            new_h = int(h_orig * scale)
            new_w = int(w_orig * scale)
            vframes = F.interpolate(vframes, size=(new_h, new_w), mode='area')
            resized = True

    h, w = vframes.shape[-2:]

    # Warmup: prepend repeated first frame
    warmup = vframes[0].unsqueeze(0).repeat(n_warmup, 1, 1, 1)
    vframes_full = torch.cat([warmup, vframes], dim=0)
    total_len = len(vframes_full)

    # Prepare mask
    mask_np = first_frame_mask.copy()
    if r_dilate > 0:
        mask_np = gen_dilate(mask_np, r_dilate, r_dilate)
    if r_erode > 0:
        mask_np = gen_erosion(mask_np, r_erode, r_erode)

    mask_t = torch.from_numpy(mask_np).float().to(device)
    if resized:
        mask_t = F.interpolate(
            mask_t.unsqueeze(0).unsqueeze(0), size=(h, w), mode='nearest'
        )[0, 0]

    objects = [1]
    results = []

    with torch.inference_mode():
        for ti in range(total_len):
            image = (vframes_full[ti] / 255.).float().to(device)
            image_np = vframes_full[ti].permute(1, 2, 0).numpy()

            if ti == 0:
                out_prob = processor.step(image, mask_t, objects=objects)
                out_prob = processor.step(image, first_frame_pred=True)
            elif ti <= n_warmup:
                out_prob = processor.step(image, first_frame_pred=True)
            else:
                out_prob = processor.step(image)

            alpha_t = processor.output_prob_to_mask(out_prob)
            pha_np  = alpha_t.unsqueeze(2).cpu().numpy()
            fgr_np  = (image_np / 255.) * pha_np  # premultiplied RGB [0,1]

            # Skip warmup frames
            if ti >= n_warmup:
                pha_out = np.round(np.clip(pha_np * 255, 0, 255)).astype(np.uint8)
                fgr_out = np.round(np.clip(fgr_np * 255, 0, 255)).astype(np.uint8)

                # Resize back to original resolution if we downsampled
                if resized:
                    pha_out = cv2.resize(pha_out[..., 0], (w_orig, h_orig),
                                        interpolation=cv2.INTER_LINEAR)[..., None]
                    fgr_out = cv2.resize(fgr_out, (w_orig, h_orig),
                                        interpolation=cv2.INTER_LINEAR)

                results.append({'pha': pha_out, 'fgr': fgr_out})

    return results


print('✅ Pipeline functions defined')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  FULL PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

import gc

# ── Read video metadata ───────────────────────────────────────────────────────
cap = cv2.VideoCapture(INPUT_VIDEO)
fps    = cap.get(cv2.CAP_PROP_FPS)
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f'Video: {total} frames @ {fps:.2f}fps  |  {width}×{height}')

# ── Step 1: Scene-cut detection ───────────────────────────────────────────────
print('\n[1/4] Detecting scene cuts ...')
cut_starts = detect_scene_cuts(INPUT_VIDEO, threshold=CUT_THRESHOLD)
# Build shot intervals: [(start, end), ...]
shot_intervals = []
for i, start in enumerate(cut_starts):
    end = cut_starts[i + 1] if i + 1 < len(cut_starts) else total
    shot_intervals.append((start, end))
print(f'Found {len(shot_intervals)} shot(s): {shot_intervals}')

# ── Step 2: Load all frames once into memory ──────────────────────────────────
# (For very long videos, consider chunked reading)
print('\n[2/4] Reading frames ...')
cap = cv2.VideoCapture(INPUT_VIDEO)
all_frames = []
for _ in tqdm(range(total), desc='Loading frames'):
    ret, frame = cap.read()
    if not ret:
        break
    all_frames.append(frame)
cap.release()
print(f'Loaded {len(all_frames)} frames')

# ── Step 3: Per-shot processing ───────────────────────────────────────────────
print('\n[3/4] Processing shots ...')
all_results = []   # list of {'pha': ..., 'fgr': ...} in frame order

for shot_idx, (shot_start, shot_end) in enumerate(shot_intervals):
    shot_frames = all_frames[shot_start:shot_end]
    n_frames    = len(shot_frames)
    print(f'\n  Shot {shot_idx + 1}/{len(shot_intervals)}: '
          f'frames {shot_start}–{shot_end - 1} ({n_frames} frames)')

    if n_frames == 0:
        continue

    # Auto-detect foreground on the first frame of this shot
    print(f'  Generating auto-mask for shot {shot_idx + 1} ...')
    first_frame_mask = get_foreground_mask(shot_frames[0], yolo_conf=YOLO_CONF)

    if first_frame_mask is None:
        print(f'  [WARN] Could not detect foreground in shot {shot_idx + 1}. '
              'Outputting blank alpha for this shot.')
        h, w = shot_frames[0].shape[:2]
        for _ in shot_frames:
            all_results.append({
                'pha': np.zeros((h, w, 1), dtype=np.uint8),
                'fgr': np.zeros((h, w, 3), dtype=np.uint8),
            })
        continue

    print(f'  Running MatAnyone 2 on {n_frames} frames ...')
    shot_results = run_matanyone2_on_shot(
        shot_frames,
        first_frame_mask,
        n_warmup=N_WARMUP,
        r_erode=R_ERODE,
        r_dilate=R_DILATE,
    )
    all_results.extend(shot_results)

    # Free GPU memory between shots
    torch.cuda.empty_cache()
    gc.collect()

print(f'\nProcessed {len(all_results)} output frames total')

# ── Step 4: Write output video ────────────────────────────────────────────────
print('\n[4/4] Writing output ...')
base_name   = os.path.splitext(os.path.basename(INPUT_VIDEO))[0]
output_path = os.path.join(OUTPUT_DIR, f'{base_name}_matte.mp4')
alpha_path  = os.path.join(OUTPUT_DIR, f'{base_name}_alpha.mp4')

GREEN = np.array([0, 177, 64], dtype=np.float32) / 255.  # broadcast green

composite_frames = []
alpha_frames     = []

for r in tqdm(all_results, desc='Compositing'):
    pha = r['pha'] / 255.     # [H, W, 1] float
    fgr = r['fgr'] / 255.    # [H, W, 3] float premultiplied

    if OUTPUT_MODE == 'greenscreen':
        comp = fgr + GREEN * (1.0 - pha)
    elif OUTPUT_MODE == 'alpha':
        comp = np.repeat(pha, 3, axis=2)
    else:  # 'transparent' — write RGBA PNGs separately below
        comp = fgr

    frame_out = np.round(np.clip(comp * 255, 0, 255)).astype(np.uint8)
    composite_frames.append(frame_out)
    alpha_frames.append(np.round(np.clip(pha * 255, 0, 255)).astype(np.uint8))

# Write composite video
imageio.mimwrite(output_path, composite_frames, fps=fps, quality=8)
imageio.mimwrite(alpha_path,  [np.repeat(a, 3, axis=2) for a in alpha_frames],
                 fps=fps, quality=8)

# Optional: write per-frame RGBA PNGs for lossless alpha
if OUTPUT_MODE == 'transparent':
    rgba_dir = os.path.join(OUTPUT_DIR, f'{base_name}_rgba_frames')
    os.makedirs(rgba_dir, exist_ok=True)
    for fi, (r, a) in enumerate(zip(composite_frames, alpha_frames)):
        rgba = np.concatenate([r, a], axis=2)   # [H, W, 4]
        Image.fromarray(rgba).save(os.path.join(rgba_dir, f'{fi:05d}.png'))
    print(f'RGBA frames saved to {rgba_dir}')

print(f'\n✅ Done!')
print(f'   Composite : {output_path}')
print(f'   Alpha     : {alpha_path}')

In [ ]:
import matplotlib.pyplot as plt
import math

n_shots  = len(shot_intervals)
n_cols   = min(4, n_shots)
n_rows   = math.ceil(n_shots / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = np.array(axes).reshape(-1)

for i, (start, end) in enumerate(shot_intervals):
    result_idx = start  # result index aligns with frame index
    if result_idx < len(all_results):
        r = all_results[result_idx]
        pha = r['pha'] / 255.
        fgr = r['fgr'] / 255.
        comp = fgr + np.array([[[0, 0.69, 0.25]]]) * (1.0 - pha)
        axes[i].imshow(np.clip(comp, 0, 1))
    axes[i].set_title(f'Shot {i+1}  (frames {start}–{end-1})', fontsize=9)
    axes[i].axis('off')

# Hide unused subplots
for j in range(n_shots, len(axes)):
    axes[j].axis('off')

plt.suptitle('First frame of each detected shot (green-screen composite)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'shot_preview.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Shot preview saved.')

In [ ]:
from google.colab import files

print(f'Downloading: {output_path}')
files.download(output_path)

In [ ]:
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(INPUT_VIDEO)
scores = []
prev_hist = None

while True:
    ret, frame = cap.read()
    if not ret:
        break
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    hist = [cv2.calcHist([hsv], [c], None, [64], [0, 256]) for c in range(3)]
    for h in hist:
        cv2.normalize(h, h)
    if prev_hist is not None:
        diffs = [cv2.compareHist(prev_hist[c], hist[c], cv2.HISTCMP_BHATTACHARYYA) for c in range(3)]
        scores.append(float(np.mean(diffs)))
    prev_hist = hist
cap.release()

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(scores, linewidth=0.8, color='steelblue', label='HSV histogram diff')
ax.axhline(CUT_THRESHOLD, color='crimson', linestyle='--', linewidth=1.2,
           label=f'Current threshold ({CUT_THRESHOLD})')
ax.set_xlabel('Frame')
ax.set_ylabel('Bhattacharyya distance')
ax.set_title('Scene cut signal — peaks above the red line are detected as cuts')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Detected cut frames at threshold {CUT_THRESHOLD}: {detect_scene_cuts(INPUT_VIDEO, CUT_THRESHOLD)}')